In [1]:
!pip install -q faster-whisper jiwer librosa soundfile tqdm num2words
!pip install -q transformers torchaudio onnxruntime==1.20.1 onnx==1.20.1 onnxruntime-gpu==1.20.1

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.0 MB/s eta 0:00:00


In [2]:
import os
import re
import torch
import torchaudio
import pandas as pd
import soundfile as sf

from tqdm import tqdm
from num2words import num2words
from faster_whisper import WhisperModel
from transformers import AutoModel

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Get the token from Kaggle Secrets
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

# Login programmatically (no interactive prompt)
login(token=secret_value_0)

In [4]:
BASE_PATH = "/kaggle/input/competitions/multilingual-speech-recognition"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [5]:
train_df = pd.read_csv(f"{BASE_PATH}/train.csv")
test_df = pd.read_csv(f"{BASE_PATH}/test.csv")

train_df["audio_path"] = train_df["audio"].apply(
    lambda x: f"{BASE_PATH}/competition_data/train/{x}"
)

test_df["audio_path"] = test_df["audio"].apply(
    lambda x: f"{BASE_PATH}/competition_data/test/{x}"
)


In [6]:
print(train_df.head)

<bound method NDFrame.head of         id            audio  \
0        0  audio_00000.wav   
1        1  audio_00001.wav   
2        2  audio_00002.wav   
3        3  audio_00003.wav   
4        4  audio_00004.wav   
...    ...              ...   
1995  1995  audio_01995.wav   
1996  1996  audio_01996.wav   
1997  1997  audio_01997.wav   
1998  1998  audio_01998.wav   
1999  1999  audio_01999.wav   

                                                   text  \
0                         you had quoted plutarch line.   
1     மலையேறுதலில் வந்து பார்த்தீங்கன்னா ஜஸ்ட்டு நம்...   
2     to do his phd in engineering about four years ...   
3                             maybe he was not at home.   
4     BUT WE DIDN'T BREAK HIS OLD WINDOW YOU KNOW EX...   
...                                                 ...   
1995  நிறைய வகையான போட்டி நடக்கும் அப்புறம் வந்து உர...   
1996  EXCLAIMED THE OTHER AS THOUGH MORE THAN SURPRI...   
1997  पाकिस्तान ने आईएमएफ़ से करीब आठ से बारह डॉलर ब...   
1998   

In [7]:
def detect_language_text(text):
    has_tamil = bool(re.search(r'[\u0B80-\u0BFF]', text))
    has_hindi = bool(re.search(r'[\u0900-\u097F]', text))
    has_english = bool(re.search(r'[a-zA-Z]', text))

    count = sum([has_tamil, has_hindi, has_english])

    if count > 1:
        return "mixed"
    elif has_tamil:
        return "tamil"
    elif has_hindi:
        return "hindi"
    elif has_english:
        return "english"
    else:
        return "unknown"


train_df["language"] = train_df["text"].apply(detect_language_text)

In [8]:
def clean_basic(text):
    text = text.lower()
    text = re.sub(r"[^\w\s\u0B80-\u0BFF\u0900-\u097F]", "", text)
    return " ".join(text.split())


def is_english(text):
    return bool(re.fullmatch(r"[a-zA-Z0-9\s]+", text))


def convert_numbers(text):
    if not is_english(text):
        return text

    def helper(match):
        try:
            return num2words(int(match.group()))
        except:
            return match.group()

    return re.sub(r"\b\d+\b", helper, text)


def finalize_text(text):
    text = clean_basic(text)
    text = convert_numbers(text)

    if is_english(text) and not text.endswith("."):
        text += "."

    return text

In [9]:
whisper_model = WhisperModel("medium", compute_type="float16")

indic_model = AutoModel.from_pretrained(
    "ai4bharat/indic-conformer-600m-multilingual",
    trust_remote_code=True
).to(DEVICE)

indic_model.eval()

config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

model_onnx.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual:
- model_onnx.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:115: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


IndicASRModel()

In [10]:
def detect_lang_whisper(model, path):
    _, info = model.transcribe(path, beam_size=1)
    return info.language


def transcribe_whisper(model, path):
    segments, _ = model.transcribe(
        path,
        beam_size=5,
        temperature=0,
        patience=0.8
    )
    return " ".join(seg.text for seg in segments).strip()


def transcribe_indic(model, path, lang):
    wav, sr = torchaudio.load(path)

    # convert to mono
    wav = torch.mean(wav, dim=0, keepdim=True)

    # resample if needed
    if sr != 16000:
        wav = torchaudio.transforms.Resample(sr, 16000)(wav)

    wav = wav.to(DEVICE)

    with torch.no_grad():
        text = model(wav, lang, "ctc")

    return text.replace("|", " ").strip()

In [11]:
def process_audio(path, whisper_model, indic_model):
    lang = detect_lang_whisper(whisper_model, path)

    if lang == "en":
        text = transcribe_whisper(whisper_model, path)

    elif lang in ["hi", "ur"]:
        text = transcribe_indic(indic_model, path, "hi")

    elif lang == "ta":
        text = transcribe_indic(indic_model, path, "ta")

    else:
        text = transcribe_whisper(whisper_model, path)

    return finalize_text(text)

In [12]:
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["language"],
    random_state=42
)

In [13]:
val_predictions = []

for _, row in tqdm(val_split.iterrows(), total=len(val_split)):
    pred = process_audio(row["audio_path"], whisper_model, indic_model)
    val_predictions.append(pred)

val_split["prediction"] = val_predictions


100%|██████████| 400/400 [09:15<00:00,  1.39s/it]


In [14]:
from jiwer import wer

val_wer = wer(
    list(map(finalize_text, val_split["text"])),
    list(map(finalize_text, val_split["prediction"]))
)

print("Validation WER:", val_wer)

Validation WER: 0.12651236191478168


In [15]:
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pred = process_audio(row["audio_path"], whisper_model, indic_model)
    predictions.append(pred)

test_df["prediction"] = predictions


# SAVE
submission = test_df[["audio", "prediction"]].copy()
submission.columns = ["audio", "text"]

submission.to_csv("submission.csv", index=False)

print(" Submission file saved!")

100%|██████████| 100/100 [02:17<00:00,  1.37s/it]

 Submission file saved!
